# Validation, EDA and Gap Analysis

This notebook serves as a validation for REFLECT. The goal is to test the following:
 - Validate claims made in the paper against real pipeline and dataset.
 - Perform exploratory data analysis both to the datasets and the results.
 - From the EDA, identify gaps in the research and specific REFLECT framework.

## Experiment 1 - Local LLM Run on Full Dataset

### Imports

In [ ]:
# Imports and log-noise suppression
import os
import logging
import warnings
import pandas as pd
import json
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np

from real_world_batch_validation import run_batch_validation
from real_world_hierarchical_prompt import config_parser
from real_world_scene_graph import SceneGraph

# Reduce third-party startup noise in notebooks
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("OPEN3D_LOG_LEVEL", "Error")

# Keep only actionable logs
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("open3d").setLevel(logging.ERROR)

# Hide known non-fatal model compatibility reports
warnings.filterwarnings(
    "ignore",
    message=r".*LOAD REPORT from: roberta-base.*",
    category=UserWarning,
 )
warnings.filterwarnings(
    "ignore",
    message=r".*UNEXPECTED.*",
    category=UserWarning,
 )
warnings.filterwarnings(
    "ignore",
    message=r".*MISSING.*",
    category=UserWarning,
 )
warnings.filterwarnings(
    "ignore",
    message=r".*Unexpected keys.*",
    category=UserWarning,
 )

In [ ]:
# Define objects and helper functions for task parsing
def _summary_to_list(summary_value):
    if isinstance(summary_value, list):
        return [str(item).strip() for item in summary_value if str(item).strip()]
    if isinstance(summary_value, str):
        return [line.strip() for line in summary_value.splitlines() if line.strip()]
    return []

class Task:
    def __init__(self, id, name, general_folder_name, gt_failure_reason, gt_failure_step, object_list, actions, success_condition, json_key=None):
        self.id = id
        self.name = name
        self.general_folder_name = general_folder_name
        self.gt_failure_reason = gt_failure_reason
        self.gt_failure_step = gt_failure_step
        self.object_list = object_list
        self.actions = actions
        self.success_condition = success_condition
        self.json_key = json_key

    @classmethod
    def from_json(cls, json_key, json_data):
        return cls(
            id=json_data.get('task_idx', json_key),
            name=json_data.get('name', ''),
            general_folder_name=json_data.get('general_folder_name', ''),
            gt_failure_reason=json_data.get('gt_failure_reason', ''),
            gt_failure_step=json_data.get('gt_failure_step', ''),
            object_list=_summary_to_list(json_data.get('object_list', [])),
            actions=_summary_to_list(json_data.get('actions', [])),
            success_condition=_summary_to_list(json_data.get('success_condition', [])),
            json_key=json_key,
        )

In [ ]:
# Load tasks
tasks_base_dir = Path.cwd()
if not (tasks_base_dir / 'tasks_real_world.json').exists():
    candidate = tasks_base_dir / 'reflect' / 'real_world_code'
    if (candidate / 'tasks_real_world.json').exists():
        tasks_base_dir = candidate
with open(tasks_base_dir / 'tasks_real_world.json', 'r') as f:
        tasks_json = json.load(f)

# Tasks to list
tasks = []
for task_key, task in tasks_json.items():
    task_obj = Task.from_json(task_key, task)
    tasks.append(task_obj)


print(f"Loaded {len(tasks)} tasks from {tasks_base_dir}")

In [ ]:
base_dir = Path.cwd()
if not (base_dir / "real_world_hierarchical_prompt.py").exists():
    candidate = base_dir / "reflect" / "real_world_code"
    if candidate.exists():
        base_dir = candidate
print("Using base dir:", base_dir)
if Path.cwd() != base_dir:
    os.chdir(base_dir)
    print("Changed cwd to:", Path.cwd())

args = config_parser().parse_args(args=[])
args.obj_det = "mdetr"
args.audio_ver = 1
args.ablation_type = 0
args.mdetr_confidence_threshold = 0.9
args.force_rebuild_sg = True
args.log_level = "INFO"
args.artifact_level = "final"
args.profile = True
args.task_workers = "auto"
args.reasoning_workers = 1
args.outlier_filter_max_points = 12000  # lower for faster runs; raise for stronger denoising

selected_task_infos = [tasks_json[task.json_key] for task in tasks]
selected_task_keys = [task.json_key for task in tasks]
batch_summary = run_batch_validation(
    args=args,
    task_infos=selected_task_infos,
    task_keys=selected_task_keys,
)
print("Batch summary saved to:", base_dir / "real_world" / "state_summary" / "batch_timings.json")

summary_root = Path("real_world") / "state_summary"
graph_dir = summary_root / tasks[0].general_folder_name / "local_graphs"
print("\nGraph artifact status:")
print("summary_root:", summary_root.resolve())
if graph_dir.exists():
    for graph_file in sorted(graph_dir.glob("local_sg_*.pkl")):
        print(graph_file.name, graph_file.stat().st_size)
else:
    print("No local_graphs directory found")
    parent_dir = summary_root / tasks[0].general_folder_name
    if parent_dir.exists():
        print("Found task dir contents:", sorted(p.name for p in parent_dir.iterdir()))
    else:
        print("Task dir missing:", parent_dir)

rows = []

for task in tasks:
    task_json = tasks_json[task.json_key]
    summary_path = summary_root / task.general_folder_name
    reasoning_candidates = [
        summary_path / "reasoning.json",
        summary_path / "reasoning-wo-framework.json",
        summary_path / "reasoning-wo-sound.json",
        summary_path / "llm-direct-reasoning.json",
    ]
    reasoning_path = next((path for path in reasoning_candidates if path.exists()), reasoning_candidates[0])
    l1_summary_path = summary_path / "state_summary_L1.txt"
    l2_summary_path = summary_path / "state_summary_L2.txt"
    local_graphs_path = summary_path / "local_graphs"

    if not reasoning_path.exists():
        print(f"Reasoning file not found for Task {task.id} at {reasoning_path}")
        continue

    with open(reasoning_path, "r") as f:
        reasoning_data = json.load(f)

    l1_summary_list = _summary_to_list(reasoning_data.get("l1_summary"))
    l2_summary_list = _summary_to_list(reasoning_data.get("l2_summary"))

    if not l1_summary_list and l1_summary_path.exists():
        with open(l1_summary_path, "r") as f:
            l1_summary_list = [line.strip() for line in f.readlines() if line.strip()]

    if not l2_summary_list and l2_summary_path.exists():
        with open(l2_summary_path, "r") as f:
            l2_summary_list = [line.strip() for line in f.readlines() if line.strip()]

    local_graph_files = []
    local_graphs = []
    if local_graphs_path.exists():
        graph_files = sorted(
            local_graphs_path.glob("local_sg_*.pkl"),
            key=lambda p: int(p.stem.split("_")[-1])
        )
        for graph_file in graph_files:
            local_graph_files.append(str(graph_file))
            with open(graph_file, "rb") as f:
                local_graphs.append(pickle.load(f))
    else:
        print(f"Local graphs directory not found for Task {task.id} at {local_graphs_path}")

    rows.append(
        {
            "task_id": task.id,
            "folder": task.general_folder_name,
            "pred_failure_reason": reasoning_data.get("pred_failure_reason"),
            "pred_failure_step": reasoning_data.get("pred_failure_step"),
            "gt_failure_reason": reasoning_data.get("gt_failure_reason"),
            "gt_failure_step": reasoning_data.get("gt_failure_step"),
            "reasoning_file": str(reasoning_path),
            "L1_summary_list": l1_summary_list,
            "L2_summary_list": l2_summary_list,
            "local_graph_files": local_graph_files,
            "local_graphs": local_graphs,
        }
    )

df_results = pd.DataFrame(rows)

In [ ]:
results = df_results.copy()  # Keep original df_results intact for any future reference

def ts_to_sec(ts):
    """Convert 'MM:SS' timestamp string to total seconds (int)."""
    m, s = ts.strip().split(":")
    return int(m) * 60 + int(s)

def _flatten_steps(val):
    """Normalise gt/pred_failure_step to a flat list[str].
    Handles: None, int/float (raw step count), str, list[str], list[list[str]].
    """
    if val is None:
        return []
    if isinstance(val, (int, float)):
        # Raw step number stored before convert_step_to_timestep was applied;
        # keep as string so downstream ts_to_sec fails gracefully (returns False)
        return [str(val)]
    if isinstance(val, str):
        return [val]
    flat = []
    for item in val:
        if isinstance(item, list):
            flat.extend(str(x) for x in item)
        else:
            flat.append(str(item))
    return flat

def _loc_hit(pred_list, gt_list):
    if not pred_list or not gt_list:
        return False
    try:
        p_sec = ts_to_sec(pred_list[0])
        gt_secs = [ts_to_sec(t) for t in gt_list]
        if len(gt_secs) == 1:
            return p_sec == gt_secs[0]
        else:
            return gt_secs[0] <= p_sec <= gt_secs[-1]
    except Exception:
        return False

# Apply _flatten_steps to gt/pred_failure_step columns
results["gt_failure_step"] = results["gt_failure_step"].apply(_flatten_steps)
results["pred_failure_step"] = results["pred_failure_step"].apply(_flatten_steps)
results["loc_hit"] = results.apply(lambda row: _loc_hit(row["pred_failure_step"], row["gt_failure_step"]), axis=1)

In [ ]:
# Create csv for failure explanation analysis
csv_columns = [
    "task",
    "pred_failure_reason",
    "gt_failure_reason",
]
exp_table = results[["folder", "pred_failure_reason", "gt_failure_reason"]].copy()
exp_table.rename(columns={"folder": "task"}, inplace=True)
exp_table.insert(len(exp_table.columns), 'human_score', pd.NA)

EXP_EVAL_PATH = base_dir / "real_world" / "failure_explanation_evaluation.csv"
exp_table.to_csv(EXP_EVAL_PATH, index=False)
print(f"\nFailure explanation evaluation table saved to: {EXP_EVAL_PATH}")
exp_table.head()

In [ ]:
# Load explanation results
explanation_results = pd.read_csv(EXP_EVAL_PATH)
explanation_results = explanation_results.dropna(subset=['human_score'])
explanation_results["human_score"] = explanation_results["human_score"].astype(int)
explanation_results.head()

In [ ]:
# Plot localization performance as % of total tasks and explanataion as % of total tasks
loc_accuracy = results["loc_hit"].mean() * 100
exp_accuracy = explanation_results["human_score"].mean() * 100

plt.bar(["Localization", "Explanation"], [loc_accuracy, exp_accuracy], color=['blue', 'orange'])
for i, v in enumerate([loc_accuracy, exp_accuracy]):
    plt.text(i, v + 1, f"{v:.1f}%", ha='center', fontweight='bold')
plt.ylim(0, 100)
plt.ylabel("Accuracy (%)")
plt.title("Performance")
plt.show()

In [ ]:
# Store results with explanations for further analysis
final_results_path = base_dir / "real_world" / "final_results_with_explanations.csv"
final_df = results.merge(explanation_results[["task", "human_score"]], left_on="folder", right_on="task", how="left").drop(columns=["task"])
final_df.to_csv(final_results_path, index=False)
print(f"\nFinal results with explanations saved to: {final_results_path}")

In [ ]:
# Results side by side with claimes
reflect_claimed_exp = 73.7
reflect_claimed_loc = 86.2

# Barplot of claimed vs actual + visualized loss (gap)
metrics = ['Loc', 'Exp']
metrics_full = ['Failure Localization Accuracy (Loc)', 'Failure Explanation Quality (Exp)']
x = np.arange(len(metrics))
w = 0.34

claimed_values = np.array([reflect_claimed_loc, reflect_claimed_exp], dtype=float)
actual_values  = np.array([loc_accuracy, exp_accuracy], dtype=float)
loss_values    = claimed_values - actual_values

fig, ax = plt.subplots(figsize=(12, 7))

# Bars
bars_claimed = ax.bar(
    x - w/2, claimed_values, w,
    color=['#4c72b0', '#e07b39', 'mediumseagreen'],
    alpha=0.65, hatch='//', edgecolor='black', linewidth=0.6,
    label='Claimed'
)
bars_actual = ax.bar(
    x + w/2, actual_values, w,
    color=['#4c72b0', '#e07b39', 'mediumseagreen'],
    label='Actual'
)

# Value labels on bars
for i, v in enumerate(claimed_values):
    ax.text(i - w/2, v + 1, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=9)
for i, v in enumerate(actual_values):
    ax.text(i + w/2, v + 1, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=9)

# Gap markers + loss labels
for i, (c, a, d) in enumerate(zip(claimed_values, actual_values, loss_values)):
    # connector showing distance between claimed and actual
    ax.plot([i - w/2, i + w/2], [c, a], color='crimson', lw=2)
    # midpoint label
    y_mid = (c + a) / 2
    ax.text(i, y_mid, f'Δ {d:.1f} pp', color='crimson', ha='center', va='bottom',
            fontsize=9, fontweight='bold',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1.5))

ax.set_xticks(x)
ax.set_xticklabels(metrics_full)
ax.set_ylabel('Percentage (%)')
ax.set_title('Theirs (Claimed) vs Our Performance by Metric')
ax.set_ylim(0, 105)
legend_handles = [
    Patch(facecolor='white', edgecolor='black', hatch='//',label='Theirs (hatched)'),
    Patch(facecolor='white', edgecolor='black', label='Ours (Solid)'),
]
ax.legend(handles=legend_handles, loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()